In [1]:
# FEATURE SELECTION & MODEL CREATION

In [2]:
## 1. Load data

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
candidates = [Path('wine_raw.csv'), Path('../01_Data_Collection/wine_raw.csv'), Path('01_Data_Collection/wine_raw.csv'), Path('Wine_ML_Assignment/01_Data_Collection/wine_raw.csv')]
path = next((p for p in candidates if p.exists()), None)
if path is not None:
    df = pd.read_csv(path)
else:
    from sklearn.datasets import load_wine
    wine = load_wine(as_frame=True)
    df = wine.frame.rename(columns={'target':'class'})
print('Dataset:', path if path else 'scikit-learn UCI Wine fallback')
print('Shape:',df.shape)
display(df.head())

Dataset: wine_raw.csv
Shape: (178, 14)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,class
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [4]:
## 2. Configure feature selection and model search

In [5]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
X=df.drop(columns=['class']);y=df['class']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
models={
 'Logistic Regression':(LogisticRegression(max_iter=3000,random_state=42),{'select__k':[5,10,13],'model__C':[0.1,1,10]}),
 'SVM':(SVC(),{'select__k':[5,10,13],'model__C':[0.1,1,10],'model__kernel':['linear','rbf']}),
 'Random Forest':(RandomForestClassifier(random_state=42),{'select__k':[5,10,13],'model__n_estimators':[100,200],'model__max_depth':[None,5]})
}
def fit_models(y_training):
 results=[];best=None
 for name,(est,params) in models.items():
  pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('select',SelectKBest(f_classif)),('model',est)])
  search=GridSearchCV(pipe,params,scoring='f1_macro',cv=cv,n_jobs=-1,refit=True)
  search.fit(X_train,y_training)
  results.append({'model':name,'cv_macro_f1':round(search.best_score_,4),'best_params':str(search.best_params_)})
  if best is None or search.best_score_>best.best_score_:best=search
 return pd.DataFrame(results),best

In [6]:
## Method A — Input X preprocessing only

In [7]:
results_A,best_A=fit_models(y_train)
display(results_A)
print('Best model A:',type(best_A.best_estimator_.named_steps['model']).__name__)
print('Best CV macro F1:',round(best_A.best_score_,4))

,model,cv_macro_f1,best_params
0,Logistic Regression,0.9796,"{'model__C': 1, 'select__k': 13}"
1,SVM,0.9927,"{'model__C': 10, 'model__kernel': 'rbf', 'sele..."
2,Random Forest,0.9789,"{'model__max_depth': None, 'model__n_estimator..."


Best model A: SVC
Best CV macro F1: 0.9927


In [8]:
## Method B — Input X and output y preprocessing

In [9]:
label_encoder=LabelEncoder()
y_train_encoded=label_encoder.fit_transform(y_train)
results_B,best_B=fit_models(y_train_encoded)
display(results_B)
print('Original labels:',list(label_encoder.classes_))
print('Best CV macro F1:',round(best_B.best_score_,4))

,model,cv_macro_f1,best_params
0,Logistic Regression,0.9796,"{'model__C': 1, 'select__k': 13}"
1,SVM,0.9927,"{'model__C': 10, 'model__kernel': 'rbf', 'sele..."
2,Random Forest,0.9789,"{'model__max_depth': None, 'model__n_estimator..."


Original labels: [np.int64(0), np.int64(1), np.int64(2)]
Best CV macro F1: 0.9927


In [10]:
## 3. Compare methods (cross-validation only)

In [11]:
comparison=results_A[['model','cv_macro_f1']].merge(results_B[['model','cv_macro_f1']],on='model',suffixes=('_input_only','_input_and_output'))
display(comparison)
print('The target was already encoded:',np.array_equal(y_train.to_numpy(),y_train_encoded))

,model,cv_macro_f1_input_only,cv_macro_f1_input_and_output
0,Logistic Regression,0.9796,0.9796
1,SVM,0.9927,0.9927
2,Random Forest,0.9789,0.9789


The target was already encoded: True


In [12]:
#3 SAVE MODEL

In [13]:
import joblib
chosen=best_A if best_A.best_score_>=best_B.best_score_ else best_B
output=Path('wine_corrected_final_model.joblib')
joblib.dump({'pipeline':chosen.best_estimator_,'label_encoder':None if chosen is best_A else label_encoder,'features':X.columns.tolist()},output)
print('Saved:',output.resolve())

Saved: C:\HOPE AI\AI TAMIL COURSE\ML & DATA SCIENCE  ADVANCED TECHNIQUES\4.Feature Selection & Model Creation\wine_corrected_final_model.joblib
